In [ ]:
# No external dependencies needed -- standard library only
print('No installs required.')

In [ ]:
import json
import os

print('Imports OK')

In [ ]:
# CONFIGURATION

TEXT_EMB_PATH  = "interactions_text_embeddings.jsonl"
IMAGE_EMB_PATH = "interactions_image_embeddings.jsonl"
OUTPUT_PATH    = "interactions_multimodal.jsonl"
PROGRESS_PATH  = "merge_progress.json"

# Expected embedding dimension -- used to validate records before writing.
# Must match the models used in both embedding notebooks:
#   BERT notebook : sentence-transformers/all-mpnet-base-v2  --> 768
#   ViT  notebook : openai/clip-vit-base-patch32             --> 768
EXPECTED_DIM = 768

print(f"Text embeddings  : {TEXT_EMB_PATH}")
print(f"Image embeddings : {IMAGE_EMB_PATH}")
print(f"Output           : {OUTPUT_PATH}")
print(f"Expected emb dim : {EXPECTED_DIM}")

In [ ]:
# PRE-FLIGHT CHECKS
# Verify both input files exist and are non-empty before doing any work.

def count_lines(path: str) -> int:
    count = 0
    with open(path) as f:
        for line in f:
            if line.strip():
                count += 1
    return count

all_ok = True

for label, path in [
    ("Text embeddings ", TEXT_EMB_PATH),
    ("Image embeddings", IMAGE_EMB_PATH),
]:
    if not os.path.exists(path):
        print(f"  [MISSING]  {label}: {path}")
        all_ok = False
    else:
        n = count_lines(path)
        size_mb = os.path.getsize(path) / (1024 ** 2)
        print(f"  [OK]  {label}: {path}  ({n:,} records, {size_mb:.1f} MB)")

if not all_ok:
    raise FileNotFoundError(
        "One or more input files are missing. "
        "Run the BERT and ViT embedding notebooks first."
    )

print("\nPre-flight checks passed. Ready to merge.")

In [ ]:
# RECORD VALIDATION HELPERS

def validate_embedding(emb, label: str, expected_dim: int) -> bool:
    if not isinstance(emb, list):
        print(f"  [WARN] {label} is not a list (got {type(emb).__name__})")
        return False
    if len(emb) != expected_dim:
        print(f"  [WARN] {label} has wrong dim: expected {expected_dim}, got {len(emb)}")
        return False
    return True


def validate_record(record: dict) -> bool:
    for field in ["user_id", "asin"]:
        if not record.get(field):
            print(f"  [WARN] Missing or empty field: '{field}'")
            return False
    rating = record.get("rating")
    try:
        if rating is None or not (1.0 <= float(rating) <= 5.0):
            print(f"  [WARN] Invalid rating: {rating}")
            return False
    except (TypeError, ValueError):
        return False
    if not validate_embedding(record.get("text_embedding"),  "text_embedding",  EXPECTED_DIM):
        return False
    if not validate_embedding(record.get("image_embedding"), "image_embedding", EXPECTED_DIM):
        return False
    return True


print("Validation helpers defined.")

In [ ]:
# LOAD IMAGE EMBEDDINGS INTO MEMORY INDEX
# We load image embeddings first because they are the smaller set
# (not every review has an image). Text embeddings are then streamed
# line by line and looked up against this index -- avoiding loading
# both large files into memory at the same time.

print("Loading image embeddings into index...")

image_index      = {}   # (user_id, asin) --> image record dict
image_dim_errors = 0

with open(IMAGE_EMB_PATH) as f:
    for i, line in enumerate(f):
        line = line.strip()
        if not line:
            continue
        try:
            record = json.loads(line)
        except json.JSONDecodeError:
            print(f"  [WARN] Malformed JSON at line {i+1} in image file -- skipped")
            continue

        uid  = record.get("user_id")
        asin = record.get("asin")
        emb  = record.get("image_embedding")

        if not uid or not asin:
            continue

        if not validate_embedding(emb, f"image_embedding (line {i+1})", EXPECTED_DIM):
            image_dim_errors += 1
            continue

        key     = (uid, asin)
        new_ts  = record.get("timestamp") or 0

        # Guard: if the same (user, asin) appears more than once in the
        # image file (should not happen after preprocessing), keep the
        # record with the later timestamp.
        if key in image_index:
            existing_ts = image_index[key].get("timestamp") or 0
            if new_ts <= existing_ts:
                continue

        image_index[key] = record

print(f"  Image records indexed    : {len(image_index):,}")
if image_dim_errors:
    print(f"  Dimension errors skipped : {image_dim_errors}")
print("Index ready.")

In [ ]:
# PROGRESS TRACKER
# Tracks which (user_id, asin) pairs have already been written so that
# rerunning the notebook never appends duplicate records to the output file.
# On a fresh run the progress file does not exist -- the merge starts clean.
# On a resumed run, already-written keys are loaded and skipped in the loop.

def load_progress() -> set:
    if not os.path.exists(PROGRESS_PATH):
        return set()
    with open(PROGRESS_PATH) as f:
        data = json.load(f)
    # Keys are stored as lists in JSON -- convert back to tuples
    return {tuple(k) for k in data.get("merged_keys", [])}


def save_progress(merged_keys: set):
    with open(PROGRESS_PATH, "w") as f:
        json.dump({"merged_keys": [list(k) for k in merged_keys]}, f)


already_merged = load_progress()

if already_merged:
    print(f"Resuming merge. Already written : {len(already_merged):,} records.")
    print("Output file will be opened in APPEND mode.")
    file_mode = "a"
else:
    print("Fresh merge run. Output file will be created from scratch.")
    file_mode = "w" 

In [ ]:
# MAIN MERGE LOOP
# Streams text embeddings line by line, looks up the matching image embedding
# by (user_id, asin) key, validates the merged record, and writes it out.
# Progress is checkpointed every SAVE_PROGRESS_EVERY records so a crash
# mid-run can be resumed without rewriting already-completed records.

SAVE_PROGRESS_EVERY = 1000

stats = {
    "text_records_read"     : 0,
    "skipped_no_image"      : 0,
    "skipped_already_done"  : 0,
    "skipped_invalid"       : 0,
    "written"               : 0,
}

merged_keys = set(already_merged)

with open(OUTPUT_PATH, file_mode) as out_f:
    with open(TEXT_EMB_PATH) as text_f:
        for i, line in enumerate(text_f):
            line = line.strip()
            if not line:
                continue

            # -- Parse text record ---------------------------------------------
            try:
                text_record = json.loads(line)
            except json.JSONDecodeError:
                print(f"  [WARN] Malformed JSON at line {i+1} in text file -- skipped")
                continue

            stats["text_records_read"] += 1

            uid  = text_record.get("user_id")
            asin = text_record.get("asin")
            key  = (uid, asin)

            # -- Skip if already written in a previous run --------------------
            if key in already_merged:
                stats["skipped_already_done"] += 1
                continue

            # -- Look up matching image embedding -----------------------------
            image_record = image_index.get(key)
            if image_record is None:
                stats["skipped_no_image"] += 1
                continue

            # -- Build merged record ------------------------------------------
            merged = {
                "user_id"        : uid,
                "asin"           : asin,
                "parent_asin"    : text_record.get("parent_asin"),
                "category"       : text_record.get("category"),
                "rating"         : text_record.get("rating"),
                "timestamp"      : text_record.get("timestamp"),
                "text_embedding" : text_record.get("text_embedding"),
                "image_embedding": image_record.get("image_embedding"),
            }

            # -- Validate before writing ---------------------------------------
            if not validate_record(merged):
                stats["skipped_invalid"] += 1
                continue

            # -- Write and track ----------------------------------------------
            out_f.write(json.dumps(merged) + "\n")
            stats["written"] += 1
            merged_keys.add(key)

            # -- Checkpoint progress periodically -----------------------------
            if stats["written"] % SAVE_PROGRESS_EVERY == 0:
                out_f.flush()
                save_progress(merged_keys)
                print(f"  Checkpoint: {stats['written']:,} records written...")

# Final progress save
save_progress(merged_keys)

print(f"\n{'='*55}")
print(f"  Text records read          : {stats['text_records_read']:>10,}")
print(f"  Skipped (no image match)   : {stats['skipped_no_image']:>10,}")
print(f"  Skipped (already written)  : {stats['skipped_already_done']:>10,}")
print(f"  Skipped (failed validation): {stats['skipped_invalid']:>10,}")
print(f"  Written to output          : {stats['written']:>10,}")
print(f"{'='*55}")
print(f"\nOutput --> {OUTPUT_PATH}")

In [ ]:
# VERIFICATION
# Reads the first 5 records of the output file and confirms the schema,
# embedding dimensions, and rating ranges look correct.

print(f"Verifying output: {OUTPUT_PATH}\n")

sample = []
with open(OUTPUT_PATH) as f:
    for i, line in enumerate(f):
        if i >= 5:
            break
        sample.append(json.loads(line.strip()))

for i, rec in enumerate(sample):
    text_dim  = len(rec.get("text_embedding",  []))
    image_dim = len(rec.get("image_embedding", []))
    t_ok  = "OK" if text_dim  == EXPECTED_DIM else "WRONG"
    i_ok  = "OK" if image_dim == EXPECTED_DIM else "WRONG"
    print(f"Record {i+1}:")
    print(f"  user_id         : {rec['user_id']}")
    print(f"  asin            : {rec['asin']}")
    print(f"  category        : {rec.get('category')}")
    print(f"  rating          : {rec['rating']}")
    print(f"  text_emb  dim   : {text_dim}  [{t_ok}]")
    print(f"  image_emb dim   : {image_dim}  [{i_ok}]")
    print()

total   = sum(1 for line in open(OUTPUT_PATH) if line.strip())
size_mb = os.path.getsize(OUTPUT_PATH) / (1024 ** 2)
print(f"Total records in output : {total:,}")
print(f"Output file size        : {size_mb:.1f} MB")
print(f"\nReady for sentiment prototype learning.")

In [ ]:
# RESET UTILITY
# Run this cell ONLY if you want to redo the merge completely from scratch.
# Deletes the output file and the progress file.
# Do NOT run accidentally -- it cannot be undone.

def reset_merge():
    for path in [OUTPUT_PATH, PROGRESS_PATH]:
        if os.path.exists(path):
            os.remove(path)
            print(f"Deleted : {path}")
        else:
            print(f"Not found (already clean) : {path}")
    print("\nReset complete. Next run will start from scratch.")

# Uncomment the line below and run this cell to reset:
# reset_merge()

print("Reset utility ready. Uncomment reset_merge() to use it.")